In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('/kaggle/input/datasets/rohan0301/unsupervised-learning-on-country-data/Country-data.csv')
dd = pd.read_csv('/kaggle/input/datasets/rohan0301/unsupervised-learning-on-country-data/data-dictionary.csv')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData Dictionary:")
print(dd.to_string(index=False))
print("\nMissing Values:\n", df.isnull().sum())
print("\nFirst 5 rows:")
df.head()

In [ ]:
print("=== Statistical Summary ===")
df.describe().T.round(2)

In [ ]:
features = df.columns[1:].tolist()  # all except 'country'

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(features):
    axes[i].hist(df[col], bins=30, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(f'{col}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')

plt.suptitle('Feature Distributions', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
features = df.columns[1:].tolist()

plt.figure(figsize=(12, 8))
corr = df[features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0,
            linewidths=0.5, annot_kws={"size": 10})

plt.title('Feature Correlation Heatmap', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
features = df.columns[1:].tolist()

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(features):
    axes[i].boxplot(df[col], patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.6))
    axes[i].set_title(f'{col}', fontsize=11, fontweight='bold')

plt.suptitle('Boxplots — Outlier Detection', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler

features = df.columns[1:].tolist()

X = df[features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=features)

print("Original Data Range (gdpp):", df['gdpp'].min(), "—", df['gdpp'].max())
print("Scaled Data Range  (gdpp):", round(X_scaled_df['gdpp'].min(),2), "—", round(X_scaled_df['gdpp'].max(),2))
print("\nScaling done! Shape:", X_scaled_df.shape)
X_scaled_df.head()

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"PC1 explains: {pca.explained_variance_ratio_[0]:.2%}")
print(f"PC2 explains: {pca.explained_variance_ratio_[1]:.2%}")
print(f"Total variance explained: {sum(pca.explained_variance_ratio_):.2%}")

pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
pca_df['Country'] = df['country'].values
pca_df.head()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

features = df.columns[1:].tolist()

inertia = []
sil_scores = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertia.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertia, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=3, color='red', linestyle='--', label='Optimal K=3')
axes[0].set_title('Elbow Method', fontsize=14, fontweight='bold')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia')
axes[0].legend()

axes[1].plot(K_range, sil_scores, 'go-', linewidth=2, markersize=8)
axes[1].axvline(x=3, color='red', linestyle='--', label='Optimal K=3')
axes[1].set_title('Silhouette Score vs K', fontsize=14, fontweight='bold')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette Score')
axes[1].legend()

plt.tight_layout()
plt.show()

best_k = K_range[np.argmax(sil_scores)]
print(f"\nBest K by Silhouette: {best_k}  |  Score: {max(sil_scores):.4f}")

In [ ]:
features = df.columns[1:].tolist()

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['KMeans_Cluster'] = kmeans.fit_predict(X_scaled)

# Profile each cluster
profile = df.groupby('KMeans_Cluster')[features].mean()

# Auto-label based on GDP per capita rank
gdpp_rank = profile['gdpp'].rank()
label_map = {}
for c in profile.index:
    if gdpp_rank[c] == 1:
        label_map[c] = 0  # Underdeveloped
    elif gdpp_rank[c] == 2:
        label_map[c] = 1  # Developing
    else:
        label_map[c] = 2  # Developed

df['Cluster_Label'] = df['KMeans_Cluster'].map(label_map)
label_names = {0: 'Underdeveloped', 1: 'Developing', 2: 'Developed'}
df['Cluster_Name'] = df['Cluster_Label'].map(label_names)

print("Cluster Distribution:")
print(df['Cluster_Name'].value_counts())
print("\nCluster Profiles (mean values):")
df.groupby('Cluster_Name')[features].mean().round(2)

In [ ]:
colors = ['#e74c3c', '#f39c12', '#2ecc71']
cluster_labels = ['Underdeveloped', 'Developing', 'Developed']

plt.figure(figsize=(12, 8))
for label_id in range(3):
    mask = df['Cluster_Label'] == label_id
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                c=colors[label_id],
                label=cluster_labels[label_id],
                alpha=0.75, s=90, edgecolors='white', linewidth=0.5)

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
plt.title('KMeans Clustering — Country Segments (PCA View)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

# Step 1: Find best eps using elbow of k-distance graph
nn = NearestNeighbors(n_neighbors=5)
nn.fit(X_scaled)
distances, _ = nn.kneighbors(X_scaled)
distances = np.sort(distances[:, 4])  # 5th neighbor distance for each point

plt.figure(figsize=(10, 4))
plt.plot(distances, color='steelblue', linewidth=2)
plt.title('K-Distance Graph (Elbow = best eps)', fontsize=13, fontweight='bold')
plt.xlabel('Points sorted by distance')
plt.ylabel('5th Nearest Neighbor Distance')
plt.tight_layout()
plt.show()

# Print distance range to choose eps manually
print(f"Min distance : {distances.min():.3f}")
print(f"25th pctile  : {np.percentile(distances, 25):.3f}")
print(f"50th pctile  : {np.percentile(distances, 50):.3f}")
print(f"75th pctile  : {np.percentile(distances, 75):.3f}")
print(f"Max distance : {distances.max():.3f}")

In [ ]:
print("Trying different eps values:\n")
print(f"{'eps':<8} {'min_samples':<14} {'Clusters':<12} {'Noise Points'}")
print("-" * 50)

results_eps = []
for eps in [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]:
    for min_s in [3, 5]:
        db_temp = DBSCAN(eps=eps, min_samples=min_s)
        labels_temp = db_temp.fit_predict(X_scaled)
        n_clust = len(set(labels_temp)) - (1 if -1 in labels_temp else 0)
        n_noise = (labels_temp == -1).sum()
        results_eps.append((eps, min_s, n_clust, n_noise))
        print(f"{eps:<8} {min_s:<14} {n_clust:<12} {n_noise}")

# Pick best: at least 2 clusters, not too much noise
valid = [(e, m, c, n) for e, m, c, n in results_eps if c >= 2 and n < len(df)*0.3]
if valid:
    best_eps, best_min_s, _, _ = valid[0]
    print(f"\nSelected eps={best_eps}, min_samples={best_min_s}")
else:
    best_eps, best_min_s = 2.0, 3
    print(f"\nFallback to eps={best_eps}, min_samples={best_min_s}")

In [ ]:
dbscan = DBSCAN(eps=best_eps, min_samples=best_min_s)
df['DBSCAN_Cluster'] = dbscan.fit_predict(X_scaled)

n_clusters = len(set(df['DBSCAN_Cluster'])) - (1 if -1 in df['DBSCAN_Cluster'].values else 0)
n_noise    = (df['DBSCAN_Cluster'] == -1).sum()

print(f"DBSCAN Clusters Found : {n_clusters}")
print(f"  Noise Points (outliers): {n_noise} ({n_noise/len(df)*100:.1f}%)")
print("\nCluster Distribution:")
print(df['DBSCAN_Cluster'].value_counts())

In [ ]:
plt.figure(figsize=(12, 8))
unique_labels = sorted(df['DBSCAN_Cluster'].unique())
palette = plt.cm.tab10.colors

for label in unique_labels:
    mask = df['DBSCAN_Cluster'] == label
    color  = 'black'      if label == -1 else palette[label % 10]
    name   = 'Noise/Outliers' if label == -1 else f'DBSCAN Cluster {label}'
    marker = 'x'          if label == -1 else 'o'
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                c=[color], label=name, alpha=0.75,
                s=90, marker=marker, edgecolors='white', linewidth=0.4)

plt.xlabel('PC1', fontsize=12)
plt.ylabel('PC2', fontsize=12)
plt.title('DBSCAN Clustering — Country Segments (PCA View)', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score

# KMeans scores
km_sil = silhouette_score(X_scaled, df['Cluster_Label'])
km_db  = davies_bouldin_score(X_scaled, df['Cluster_Label'])

# DBSCAN scores — safely handle noise & single-cluster cases
dbscan_mask   = df['DBSCAN_Cluster'] != -1
X_db_clean    = X_scaled[dbscan_mask]
labels_db_clean = df.loc[dbscan_mask, 'DBSCAN_Cluster'].values
n_unique_db   = len(set(labels_db_clean))

if n_unique_db >= 2:
    db_sil = silhouette_score(X_db_clean, labels_db_clean)
    db_db  = davies_bouldin_score(X_db_clean, labels_db_clean)
    db_sil_str = f"{db_sil:.4f}"
    db_db_str  = f"{db_db:.4f}"
else:
    db_sil = db_db = None
    db_sil_str = db_db_str = "N/A (1 cluster)"
    print("⚠️  DBSCAN produced only 1 cluster after noise removal.")
    print("    Scores cannot be computed — try adjusting eps in Cell 12b.")

# Summary table
print("=== Clustering Algorithm Comparison ===\n")
print(f"{'Metric':<35} {'KMeans':>12} {'DBSCAN':>18}")
print("-" * 67)
print(f"{'Silhouette Score (↑ better)':<35} {km_sil:>12.4f} {db_sil_str:>18}")
print(f"{'Davies-Bouldin Score (↓ better)':<35} {km_db:>12.4f} {db_db_str:>18}")

# Bar chart (only if DBSCAN scored)
if db_sil is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].bar(['KMeans', 'DBSCAN'], [km_sil, db_sil], color=['steelblue', 'coral'], edgecolor='white')
    axes[0].set_title('Silhouette Score (Higher = Better)', fontweight='bold')
    axes[0].set_ylabel('Score')

    axes[1].bar(['KMeans', 'DBSCAN'], [km_db, db_db], color=['steelblue', 'coral'], edgecolor='white')
    axes[1].set_title('Davies-Bouldin Score (Lower = Better)', fontweight='bold')
    axes[1].set_ylabel('Score')

    plt.tight_layout()
    plt.show()
else:
    # KMeans-only chart
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].bar(['KMeans'], [km_sil], color='steelblue', edgecolor='white')
    axes[0].set_title('Silhouette Score', fontweight='bold')
    axes[1].bar(['KMeans'], [km_db], color='steelblue', edgecolor='white')
    axes[1].set_title('Davies-Bouldin Score', fontweight='bold')
    plt.suptitle('KMeans Scores (DBSCAN N/A)', fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

features = df.columns[1:10].tolist()  # original 9 features only

X_cls = df[features].copy()
y_cls = df['Cluster_Label'].copy()   # 0=Underdeveloped, 1=Developing, 2=Developed

X_train, X_test, y_train, y_test = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

print(f"Train size : {X_train.shape}")
print(f"Test size  : {X_test.shape}")
print(f"\nClass distribution in train:\n{y_train.value_counts()}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

rf = RandomForestClassifier(n_estimators=200, random_state=42, max_depth=10)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("=== Random Forest Results ===\n")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['Underdeveloped','Developing','Developed']))

# Confusion Matrix
plt.figure(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Underdeveloped','Developing','Developed'],
            yticklabels=['Underdeveloped','Developing','Developed'])
plt.title('Random Forest — Confusion Matrix', fontweight='bold')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators=200, learning_rate=0.1,
                     max_depth=5, random_state=42,
                     use_label_encoder=False, eval_metric='mlogloss')
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

print("=== XGBoost Results ===\n")
print(f"Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb, target_names=['Underdeveloped','Developing','Developed']))

# Confusion Matrix
plt.figure(figsize=(7, 5))
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Underdeveloped','Developing','Developed'],
            yticklabels=['Underdeveloped','Developing','Developed'])
plt.title('XGBoost — Confusion Matrix', fontweight='bold')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
features = df.columns[1:10].tolist()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Random Forest
rf_importance = pd.Series(rf.feature_importances_, index=features).sort_values()
rf_importance.plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Random Forest — Feature Importance', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Importance Score')

# XGBoost
xgb_importance = pd.Series(xgb.feature_importances_, index=features).sort_values()
xgb_importance.plot(kind='barh', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('XGBoost — Feature Importance', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid, cv=5,
    scoring='accuracy', n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

print(f"Best Parameters : {grid_search.best_params_}")
print(f"Best CV Score   : {grid_search.best_score_:.4f}")

best_rf = grid_search.best_estimator_
y_pred_best_rf = best_rf.predict(X_test)
print(f"Tuned RF Test Accuracy: {accuracy_score(y_test, y_pred_best_rf):.4f}")

In [ ]:
results = {
    'Model': ['Random Forest', 'XGBoost', 'Tuned Random Forest'],
    'Test Accuracy': [
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb),
        accuracy_score(y_test, y_pred_best_rf)
    ]
}

results_df = pd.DataFrame(results).sort_values('Test Accuracy', ascending=False)
results_df['Test Accuracy'] = results_df['Test Accuracy'].round(4)

print("=== Final Model Comparison ===\n")
print(results_df.to_string(index=False))

plt.figure(figsize=(9, 5))
bars = plt.barh(results_df['Model'], results_df['Test Accuracy'],
                color=['#2ecc71', '#3498db', '#9b59b6'], edgecolor='white')

for bar, val in zip(bars, results_df['Test Accuracy']):
    plt.text(val - 0.05, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', color='white', fontweight='bold', fontsize=12)

plt.xlim(0.5, 1.05)
plt.title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
plt.xlabel('Test Accuracy')
plt.tight_layout()
plt.show()

In [ ]:
aid_df = df[df['Cluster_Name'] == 'Underdeveloped'][
    ['country', 'child_mort', 'income', 'gdpp', 'life_expec']
].sort_values('child_mort', ascending=False).reset_index(drop=True)

print(f"Total countries needing aid: {len(aid_df)}\n")
print(aid_df.head(15).to_string(index=False))

# Plot
top10 = aid_df.head(10)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(top10['country'], top10['child_mort'], color='#e74c3c', edgecolor='white')
axes[0].set_title('Top 10 — Highest Child Mortality', fontweight='bold')
axes[0].set_xlabel('Child Mortality Rate')
axes[0].invert_yaxis()

axes[1].barh(top10['country'], top10['gdpp'], color='#e67e22', edgecolor='white')
axes[1].set_title('Top 10 — Lowest GDP per Capita', fontweight='bold')
axes[1].set_xlabel('GDP per Capita (USD)')
axes[1].invert_yaxis()

plt.suptitle('Countries Most in Need of Aid', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()